In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_csv('salaries_cleaned.csv')

In [6]:
df_copy = df

In [7]:
df.head()

,work_year,experience_level,employment_type,job_title,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,EN,FT,Other,69120,NL,0,NL,M
1,2025,EN,FT,Other,50160,NL,0,NL,M
2,2025,EN,FT,Data Engineer,158113,US,0,US,M
3,2025,EN,FT,Data Engineer,87795,US,0,US,M
4,2025,EX,FT,Data Engineer,351410,US,0,US,M


In [8]:
train_df = pd.read_csv('train_df.csv') 
test_df = pd.read_csv('test_df.csv')

In [9]:
# Mark them so we can separate them later
train_df['dataset_source'] = 'train'
test_df['dataset_source'] = 'test'

In [10]:
# Stack them on top of each other
full_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)


### Map Ordinal Features: Manually map the features that have a clear orde


In [11]:
# Example in pandas
exp_map = {'EN': 0, 'MI': 1, 'SE': 2, 'EX': 3}
size_map = {'S': 0, 'M': 1, 'L': 2}

full_df['experience_level'] = full_df['experience_level'].map(exp_map)
full_df['company_size'] = full_df['company_size'].map(size_map)

In [12]:
# 2. Encode Nominal Features 

full_df = pd.get_dummies(full_df, columns=['employment_type', 'job_title', 'employee_residence', 'company_location'])

In [13]:
full_df.head()

,work_year,experience_level,salary_in_usd,remote_ratio,company_size,dataset_source,employment_type_CT,employment_type_FL,employment_type_FT,employment_type_PT,...,company_location_TH,company_location_TR,company_location_TW,company_location_UA,company_location_US,company_location_VE,company_location_VN,company_location_XK,company_location_ZA,company_location_ZM
0,2024,1,183300,0,1,train,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
1,2025,1,250000,100,1,train,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
2,2025,3,156160,0,1,train,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
3,2024,3,255000,0,1,train,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
4,2025,1,76000,0,1,train,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False


In [14]:
# Split them using the marker we created earlier
train_processed = full_df[full_df['dataset_source'] == 'train'].drop('dataset_source', axis=1)
test_processed = full_df[full_df['dataset_source'] == 'test'].drop('dataset_source', axis=1)

In [16]:
# Training Data
y_train = train_processed['salary_in_usd']
X_train = train_processed.drop('salary_in_usd', axis=1)

# Testing Data
y_test = test_processed['salary_in_usd']
X_test = test_processed.drop('salary_in_usd', axis=1)

print("Data loaded and processed successfully!")
print(f"Training Shape: {X_train.shape}")
print(f"Testing Shape:  {X_test.shape}")

Data loaded and processed successfully!
Training Shape: (41929, 440)
Testing Shape:  (10483, 440)


### Train a Baseline (Overfit) Model

In [17]:
# 1. Initialize the model (with no limits)
baseline_model = DecisionTreeRegressor(random_state=42)


In [18]:
# 2. Train the model
baseline_model.fit(X_train, y_train)



,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [19]:
# 3. Evaluate on BOTH training and test data
preds_train = baseline_model.predict(X_train)
preds_test = baseline_model.predict(X_test)

# RMSE (Root Mean Squared Error) is a common metric. It's just the square root of MSE.
rmse_train = np.sqrt(mean_squared_error(y_train, preds_train))
rmse_test = np.sqrt(mean_squared_error(y_test, preds_test))

print(f"Baseline Training RMSE: {rmse_train}")
print(f"Baseline Test RMSE: {rmse_test}")

Baseline Training RMSE: 62378.70867794427
Baseline Test RMSE: 67508.94418907918


In [20]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 7, 10, None],  # How deep the tree can go (None = no limit)
    'min_samples_split': [10, 20, 40], # Min samples in a node to split it
    'min_samples_leaf': [5, 10, 20]    # Min samples allowed in a leaf
}

# 2. Set up the Grid Search

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42), 
    param_grid=param_grid,  # The rules to test
    cv=5,                   # 5-fold cross-validation (it splits the training data 5 ways to test itself)
    scoring='neg_mean_squared_error',
    n_jobs=-1               # This uses all your computer's CPU cores to make it run faster
)

# 3. Run the search on your training data
# This is the part that will take a minute or two
grid_search.fit(X_train, y_train)

# 4. Get the best model and its parameters
print(f"Best parameters found: {grid_search.best_params_}")

# 5. Save the winning model
best_model = grid_search.best_estimator_

print("\nGrid Search is complete!")

Best parameters found: {'max_depth': None, 'min_samples_leaf': 5, 'min_samples_split': 40}

Grid Search is complete!


In [22]:
# We use the 'best_model' variable from the grid search
final_preds = best_model.predict(X_test)

# 2. Calculate the new RMSE
final_rmse = np.sqrt(mean_squared_error(y_test, final_preds))

# 3. Print the comparison
print("--- Model Comparison ---")
print(f"Baseline Test RMSE: {rmse_test}") # This should be 67508.94...
print(f"Final Tuned Test RMSE: {final_rmse}")

if final_rmse < rmse_test:
    print("\nSuccess! The tuned model is more accurate.")
else:
    print("\nInteresting! The baseline model was slightly better.")

--- Model Comparison ---
Baseline Test RMSE: 67508.94418907918
Final Tuned Test RMSE: 66429.39003097147

Success! The tuned model is more accurate.
